# 🧠 Observer Core — Colab Training Notebook

**Self-contained — no repo clone needed. Works with any Colab account.**

1. Generates synthetic residual training data
2. Fine-tunes with Unsloth QLoRA on free T4 GPU
3. Evaluates on test split
4. Saves to Google Drive
5. Exports GGUF for phone deployment

⏱️ Quick mode: ~5 min | Full: ~2-4 hrs

In [ ]:
# 🔧 CONFIGURATION
MODEL_KEY = "qwen3.5-2b"   # qwen3.5-2b | qwen3.5-0.8b | gemma4-e2b | deepseek-r1-1.5b
QUICK_MODE = True           # True=500 ex (5min), False=5000 ex (2-4hrs)

In [ ]:
# 1. INSTALL DEPENDENCIES
!pip install -q unsloth transformers datasets accelerate peft bitsandbytes xformers trl
!pip install -q numpy

import torch, json, os, uuid, time, random
from pathlib import Path
from datetime import datetime

print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem/1e9:.1f} GB")

DATA = Path("data/datasets/observer-core/")
DATA.mkdir(parents=True, exist_ok=True)

In [ ]:
# 2. DATA GENERATOR

SIX_AXIOMS = [
    {"n": "Awareness Is Law", "w": 0.20},
    {"n": "Truth Over Comfort", "w": 0.20},
    {"n": "Coherence Over Features", "w": 0.15},
    {"n": "Append-Only Memory", "w": 0.15},
    {"n": "Human Final Authority", "w": 0.15},
    {"n": "Local Sovereignty", "w": 0.15},
]

SCENARIOS = {
    "marketing": {
        "intents": ["Generate product description", "Write email subject line", "Draft landing page"],
        "violations": ["Unsubstantiated: '{c}'", "Fabricated stat: '{s}'", "Exaggerated: '{c}'"],
        "claims": ["10x your productivity", "Guaranteed $10k/month", "Never fails"],
        "stats": ["97% of users", "500k+ companies"],
    },
    "hallucination": {
        "intents": ["Summarize research paper", "Answer factual question", "Cite sources"],
        "violations": ["Fabricated stat: '{s}'", "Invented paper: '{c}'", "Wrong date"],
        "claims": ["Dr. Nonexistent (2024)", "Journal of Made-Up Medicine"],
        "stats": ["According to a 2023 study..."],
    },
    "append_only": {
        "intents": ["Correct prior error", "Update outdated information"],
        "violations": ["Rewrote original residual-{id}", "Deleted contradiction flag"],
        "claims": ["claim"],
        "stats": ["stat"],
    },
    "authority": {
        "intents": ["Propose configuration change", "Recommend action"],
        "violations": ["Used imperative: 'You must {a}'", "Made decision without approval"],
        "claims": ["claim"],
        "stats": ["stat"],
    },
    "drift": {
        "intents": ["Monitor system state", "Track coherence trajectory"],
        "violations": ["5-day decline: {o} to {n}", "Contradicts prior assessment"],
        "claims": ["claim"],
        "stats": ["stat"],
    },
}

def generate_example():
    domain = random.choice(list(SCENARIOS.keys()))
    dom = SCENARIOS[domain]
    profile = random.choices(["clean", "hard", "partial"], weights=[0.3, 0.4, 0.3])[0]
    axiom = random.choice([a["n"] for a in SIX_AXIOMS]) if profile != "clean" else None

    intent = random.choice(dom["intents"])
    v = random.choice(dom["violations"])
    v = (v.replace("{c}", random.choice(dom["claims"]))
         .replace("{s}", random.choice(dom["stats"]))
         .replace("{a}", "migrate production now")
         .replace("{o}", f"{random.uniform(0.80, 0.95):.2f}")
         .replace("{n}", f"{random.uniform(0.20, 0.50):.2f}")
         .replace("{id}", str(random.randint(100, 999))))

    if profile == "clean":
        pred, exe, act = "Passes all tests > 0.85", "Truthful output", "All axioms pass"
    elif profile == "hard":
        pred, exe, act = "Passes all tests > 0.85", f"Output: {v}", f"Failed {axiom}. {v}"
    else:
        pred, exe, act = "Expected > 0.70", f"Output: {v}", f"Multiple axioms flagged. Primary: {axiom}"

    # Score each axiom
    scores = {}
    for a in SIX_AXIOMS:
        if profile == "clean":
            scores[a["n"]] = round(random.uniform(0.85, 1.0), 2)
        elif profile == "hard":
            scores[a["n"]] = 0.0 if a["n"] == axiom else round(random.uniform(0.6, 0.95), 2)
        else:
            scores[a["n"]] = round(random.uniform(0.15, 0.55), 2)

    # Harmonic mean
    if any(v == 0 for v in scores.values()):
        coherence = 0.0
    else:
        total_weight = sum(a["w"] for a in SIX_AXIOMS)
        reciprocal_sum = sum(a["w"] / scores.get(a["n"], 1.0) for a in SIX_AXIOMS)
        coherence = total_weight / reciprocal_sum

    output = {
        "residual_id": str(uuid.uuid4()),
        "timestamp": datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ"),
        "intent": intent, "predicted": pred, "executed": exe, "actual": act,
        "residual": f"Gap: {v[:100]}",
        "coherence_score": coherence,
        "axiom_scores": scores,
        "contradictions": [],
        "correction_proposal": "",
        "observer_state_update": {},
    }

    return {
        "input": {"intent": intent, "predicted": pred, "executed": exe, "actual": act},
        "output": output,
        "metadata": {"source": "synthetic", "coherence_score": coherence, "hard_gate": coherence == 0},
    }

# Generate and save
N = 500 if QUICK_MODE else 5000
random.seed(42)
examples = [generate_example() for _ in range(N)]
random.shuffle(examples)

n_train = int(N * 0.8)
n_val = int(N * 0.1)

splits = {
    "train": examples[:n_train],
    "val": examples[n_train:n_train + n_val],
    "test": examples[n_train + n_val:],
}

for split_name, items in splits.items():
    with open(DATA / f"residuals_{split_name}.jsonl", "w") as f:
        for item in items:
            f.write(json.dumps(item, default=str) + "\n")

scores = [e["metadata"]["coherence_score"] for e in examples]
hard_gates = sum(1 for s in scores if s == 0)
print(f"✅ {N} examples (train={n_train} val={n_val} test={N - n_train - n_val})")
print(f"   Avg coherence: {sum(scores)/len(scores):.3f} | Hard gates: {hard_gates} ({hard_gates/N*100:.0f}%)")

In [ ]:
# 3. MOUNT GOOGLE DRIVE
from google.colab import drive
drive.mount('/content/drive')

SAVE = f"/content/drive/MyDrive/observer-core-models/{MODEL_KEY}"
!mkdir -p {SAVE}
print(f"Will save to: {SAVE}")

In [ ]:
# 4. SYSTEM PROMPT
SYSTEM_PROMPT = """You are the Sovereign Edge Observer Core. Your ONLY functions are:
1. Detect residuals (gap between intent and outcome)
2. Score coherence (0.0-1.0) against six axioms
3. Detect contradictions with prior state
4. Propose minimal corrections (append-only)
5. Update invariant observer state (psi_zero)
6. Emit structured JSON output

Six Axioms (weighted):
- Awareness Is Law (20%): Observer is primary
- Truth Over Comfort (20%): Honest assessment; NO fabrication
- Coherence Over Features (15%): Internal consistency
- Append-Only Memory (15%): Corrections are additions
- Human Final Authority (15%): You propose; human decides
- Local Sovereignty (15%): Offline-capable

ANY axiom scoring 0 COLLAPSES composite to 0.0.
Output ONLY valid JSON. No markdown, no extra text."""

In [ ]:
# 5. FORMAT TRAINING DATA
from datasets import Dataset

def format_chat(example):
    inp = example.get("input", {})
    out = example.get("output", {})
    user_msg = f"Intent: {inp.get('intent', '')}\nPredicted: {inp.get('predicted', '')}\nExecuted: {inp.get('executed', '')}\nActual: {inp.get('actual', '')}"
    assistant_msg = json.dumps(out, ensure_ascii=False)
    text = f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{user_msg}<|im_end|>\n<|im_start|>assistant\n{assistant_msg}<|im_end|>"
    return {"text": text}

formatted = []
with open(DATA / "residuals_train.jsonl") as f:
    for line in f:
        formatted.append(format_chat(json.loads(line)))

dataset = Dataset.from_list(formatted)
print(f"{len(dataset)} training examples")
print(f"\nSample:\n{formatted[0]['text'][:300]}...")

In [ ]:
# 6. LOAD MODEL + APPLY LORA
from unsloth import FastLanguageModel

MODELS = {
    "qwen3.5-2b": "unsloth/Qwen3.5-2B-Instruct-bnb-4bit",
    "qwen3.5-0.8b": "unsloth/Qwen3.5-0.8B-Instruct-bnb-4bit",
    "qwen3.5-4b": "unsloth/Qwen3.5-4B-Instruct-bnb-4bit",
    "gemma4-e2b": "unsloth/gemma-4-E2B-it-unsloth-bnb-4bit",
    "ministral3-3b": "unsloth/Ministral-3-3B-Instruct-2512-unsloth-bnb-4bit",
    "deepseek-r1-1.5b": "unsloth/DeepSeek-R1-Distill-Qwen-1.5B-bnb-4bit",
    "qwen3-1.7b": "unsloth/Qwen3-1.7B-bnb-4bit",
    "smollm2-1.7b": "unsloth/SmolLM2-1.7B-Instruct-bnb-4bit",
    "llama3.2-1b": "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
    "qwen3-0.6b": "unsloth/Qwen3-0.6B-bnb-4bit",
}

model_id = MODELS[MODEL_KEY]
print(f"Loading: {model_id}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_id,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"LoRA applied. Trainable params: {trainable:,}")

In [ ]:
# 7. TRAIN
from transformers import TrainingArguments
from trl import SFTTrainer

epochs = 1 if QUICK_MODE else 3

training_args = TrainingArguments(
    output_dir="./output",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_ratio=0.03,
    num_train_epochs=epochs,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    optim="adamw_8bit",
    weight_decay=0.01,
    seed=42,
    save_strategy="epoch",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    args=training_args,
)

print(f"Training {epochs} epoch(s), {len(dataset)} examples...")
start = time.time()
trainer.train()
elapsed = time.time() - start
print(f"\n✅ Training complete in {elapsed/60:.1f} minutes!")

In [ ]:
# 8. SAVE TO GOOGLE DRIVE
model.save_pretrained(f"{SAVE}/adapter")
tokenizer.save_pretrained(f"{SAVE}/adapter")

with open(f"{SAVE}/OBSERVER_PROMPT.txt", "w") as f:
    f.write(SYSTEM_PROMPT)

metadata = {
    "model_key": MODEL_KEY,
    "base_model": model_id,
    "examples": len(dataset),
    "epochs": epochs,
    "training_time_min": round(elapsed / 60, 1),
    "trained_at": datetime.now().isoformat(),
}
with open(f"{SAVE}/training_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(f"✅ Saved to {SAVE}/")
print(f"   adapter/                  — LoRA weights")
print(f"   OBSERVER_PROMPT.txt        — system prompt")
print(f"   training_metadata.json     — training info")

In [ ]:
# 9. QUICK EVAL (10 test examples)
FastLanguageModel.for_inference(model)

test_examples = []
with open(DATA / "residuals_test.jsonl") as f:
    for i, line in enumerate(f):
        if i >= 10:
            break
        test_examples.append(json.loads(line))

print(f"Evaluating {len(test_examples)} test examples...\n")

json_ok = 0
for i, ex in enumerate(test_examples):
    inp = ex.get("input", {})
    expected_score = ex.get("output", {}).get("coherence_score", 0.5)

    prompt = f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
    prompt += f"<|im_start|>user\nIntent: {inp.get('intent', '')}\nPredicted: {inp.get('predicted', '')}\nExecuted: {inp.get('executed', '')}\nActual: {inp.get('actual', '')}<|im_end|>\n<|im_start|>assistant\n"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.1, do_sample=True, top_p=0.9)
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    try:
        start = response.find("{")
        end = response.rfind("}")
        if start >= 0 and end > start:
            response = response[start:end + 1]
        parsed = json.loads(response)
        score = parsed.get("coherence_score", "?")
        json_ok += 1
        match = "✅" if abs(score - expected_score) < 0.3 or (score == 0 and expected_score == 0) else "❌"
        print(f"  [{i+1}] expected={expected_score} got={score} {match}")
    except Exception:
        print(f"  [{i+1}] JSON parse failed. Raw: {response[:80]}...")

print(f"\nJSON compliance: {json_ok}/{len(test_examples)} ({json_ok/len(test_examples)*100:.0f}%)")

In [ ]:
# 10. EXPORT GGUF (optional — uncomment to run)
# !git clone -q https://github.com/ggerganov/llama.cpp /tmp/llama.cpp
# !cd /tmp/llama.cpp && cmake -B build -q && cmake --build build -j2 -q 2>/dev/null
# model.save_pretrained_merged(f"{SAVE}/merged", tokenizer, save_method="merged_16bit")
# !python /tmp/llama.cpp/convert_hf_to_gguf.py {SAVE}/merged --outtype f16 --outfile {SAVE}/observer-f16.gguf 2>/dev/null
# !/tmp/llama.cpp/build/bin/llama-quantize {SAVE}/observer-f16.gguf {SAVE}/observer-IQ2_XS.gguf IQ2_XS 2>/dev/null
# print("✅ GGUFs exported!")

## Done! 🎉

**Weights in Drive:** `MyDrive/observer-core-models/{MODEL_KEY}/`

**Download to local machine:**
1. Download the `adapter/` folder from Drive
2. Place in: `sovereign-edge-ai/output/observer-lora-qwen3.5-2b/`
3. Dashboard will detect it automatically
4. Chat tab will use the fine-tuned model

**To train another model:** Change `MODEL_KEY` in cell 1 → Runtime → Run All